**2. Feature Engineering**
- A. Machine baseline
- B. Deviations (how unusual is it from normal?)
- C. Z-score (anomaly strength)
- D. Rolling behavior

In [0]:
df = spark.table("machine_cleaned_data")


from pyspark.sql.window import Window
from pyspark.sql.functions import avg, stddev

#Split the dataset into groups by machine_id
machine_window = Window.partitionBy("machine_id")   

#Calculate the mean and standard deviation for each group(each machine type) which are machine baseline features
df_feat = df \
    .withColumn("temp_mean_m", avg("temperature").over(machine_window)) \
    .withColumn("temp_std_m", stddev("temperature").over(machine_window)) \
    .withColumn("vib_mean_m", avg("vibration").over(machine_window)) \
    .withColumn("vib_std_m", stddev("vibration").over(machine_window)) \
    .withColumn("energy_mean_m", avg("energy_consumption").over(machine_window)) \
    .withColumn("energy_std_m", stddev("energy_consumption").over(machine_window)) \
    .withColumn("pressure_mean_m", avg("pressure").over(machine_window)) \
    .withColumn("pressure_std_m", stddev("pressure").over(machine_window))

df_feat.show(5)


In [0]:
from pyspark.sql.functions import col

#How far is current value from mean (find differences to detect overheating, abnormal vibration, energy spikes)
df_feat = df_feat \
    .withColumn("temp_dev", col("temperature") - col("temp_mean_m")) \
    .withColumn("vib_dev", col("vibration") - col("vib_mean_m")) \
    .withColumn("energy_dev", col("energy_consumption") - col("energy_mean_m")) \
    .withColumn("pressure_dev", col("pressure") - col("pressure_mean_m"))

In [0]:
#deviation alone is not enough. 
# Z_score = deviation / natural variation
#“Is this deviation big compared to how unstable this machine usually is?” Exap: normal machine ->+5 daviation is Not suspicious but stable machine->+5 deviation is suspicious
from pyspark.sql.functions import col, abs

df_feat = df_feat \
    .withColumn("temp_z", abs(col("temp_dev")) / col("temp_std_m")) \
    .withColumn("vib_z", abs(col("vib_dev")) / col("vib_std_m")) \
    .withColumn("energy_z", abs(col("energy_dev")) / col("energy_std_m")) \
    .withColumn("pressure_z", abs(col("pressure_dev")) / col("pressure_std_m"))


#Let's see how many machines are suspicious


df_feat.show(5)


In [0]:
#“Is this getting worse over time?”
#“Is this a short spike or a trend?”

from pyspark.sql.functions import avg, stddev, abs, col, lag
from pyspark.sql.window import Window


# =========================
# 1. TIME WINDOWS
# =========================

# Ordered time window (for lag)
#each machine is treated separately and rows are sorted by time
time_window = Window.partitionBy("machine_id").orderBy("timestamp")

# Rolling window (last 5 records)
rolling_window = Window.partitionBy("machine_id") \
    .orderBy("timestamp") \
    .rowsBetween(-5, 0)


# =========================
# 2. REUSABLE FUNCTION
# =========================

def add_sensor_features(df, sensor):

    prev_col = f"{sensor}_prev"
    change_col = f"{sensor}_change"
    change_abs_col = f"{sensor}_change_abs"
    roll_mean_col = f"{sensor}_roll_mean"
    roll_std_col = f"{sensor}_roll_std"

    # lag (previous value)
    df = df.withColumn(prev_col, lag(sensor).over(time_window))

    # change rate
    df = df.withColumn(change_col, col(sensor) - col(prev_col))

    # absolute change (important for anomaly detection)
    df = df.withColumn(change_abs_col, abs(col(change_col)))

    # rolling mean (trend)
    df = df.withColumn(roll_mean_col, avg(sensor).over(rolling_window))

    # rolling std (instability)
    df = df.withColumn(roll_std_col, stddev(sensor).over(rolling_window))

    return df


# =========================
# 3. APPLY TO ALL SENSORS
# =========================

sensors = [
    "temperature",
    "vibration",
    "humidity",
    "pressure",
    "energy_consumption"
]



for s in sensors:
    df_feat = add_sensor_features(df_feat, s)


# =========================
# 4. CLEAN NULLS (IMPORTANT)
# =========================

df_feat = df_feat.fillna(0)


# =========================
# 5. VIEW RESULT
# =========================



In [0]:
df_feat.write.mode("overwrite").saveAsTable("feature_data")